# Kubernetes 第2周：网络与存储

> **学习目标**：理解 K8s 网络模型，配置 Ingress 路由，管理持久化存储

---

## K8s 网络模型

K8s 网络遵循三个基本原则：

1. **所有 Pod 可以互相通信（无需 NAT）** — 不管在哪个节点
2. **所有节点可以与所有 Pod 通信** — 无需 NAT
3. **Pod 看到的自己的 IP 和其他 Pod 看到的一致** — 没有地址转换

这看起来简单，但实现复杂——需要 CNI（Container Network Interface）插件来实现。

```
Node 1 (10.0.1.0/24)              Node 2 (10.0.2.0/24)
┌─────────────────────┐          ┌─────────────────────┐
│ Pod A: 10.244.1.5   │          │ Pod B: 10.244.2.7   │
│ Pod C: 10.244.1.6   │ ◄──────► │ Pod D: 10.244.2.8   │
└─────────────────────┘   CNI    └─────────────────────┘
      跨节点通信由 CNI 插件负责（Calico/Flannel/Cilium）
```

| CNI 插件 | 特点 | 适用场景 |
|----------|------|----------|
| **Flannel** | 简单，只做网络互通 | 小集群、学习 |
| **Calico** | 高性能，支持 NetworkPolicy | 生产环境 |
| **Cilium** | eBPF，可观测性强 | 大规模、安全敏感 |

---

## 四种网络通信场景

```
1. 同 Pod 内容器通信    → 共享 localhost（同一网络命名空间）
2. 同节点 Pod 间通信     → 通过虚拟网桥（cni0 / docker0）
3. 跨节点 Pod 间通信     → CNI 插件封装转发（VXLAN / IPIP / BGP）
4. Pod 访问外部网络       → 通过节点 iptables SNAT
```

---

## DNS：服务发现的核心

K8s 集群内部有一个 CoreDNS 服务。**每个 Service 自动获得一个 DNS 记录**：

```
<service-name>.<namespace>.svc.cluster.local

# 例如：
py-api-svc.default.svc.cluster.local  → 完整域名
py-api-svc.default                    → 同 namespace 可省略后缀
py-api-svc                            → 同 namespace 直接用服务名
```

Pod 也是同样的规则，只不过 Pod 的 DNS 名用 Pod IP 中的点划线代替：
```
10-244-1-5.default.pod.cluster.local
```

In [ ]:
# 验证 DNS 解析
# 先创建一个 Service 和一个测试 Pod
! kubectl create deployment dns-test --image=python:3.12-slim --replicas=1 -- \
  python -c "from http.server import HTTPServer,BaseHTTPRequestHandler as H;HTTPServer(('',8000),H).serve_forever()" 2>/dev/null
! kubectl expose deployment dns-test --port=80 --target-port=8000 2>/dev/null

# 用 dnsutils 镜像测试 DNS 解析
! kubectl run dnsutils --image=registry.k8s.io/e2e-test-images/jessie-dnsutils:3.5 --rm -it --restart=Never -- \
  nslookup dns-test 2>/dev/null || echo "kind 环境下 DNS 测试可能需要调整"

# 清理
! kubectl delete deployment dns-test --wait=false 2>/dev/null
! kubectl delete service dns-test --wait=false 2>/dev/null
print("DNS 原理已演示")

---

## Service 深入

### Service 是如何工作的？

当你访问 Service 的 ClusterIP:Port 时，实际发生了什么：

```
1. 你的请求 → Service ClusterIP:Port
2. kube-proxy 在节点上配置 iptables 规则
3. iptables 随机选择一个后端 Pod IP
4. DNAT：目标地址被改写为 Pod IP
5. Pod 处理请求并响应
```

### Headless Service

设置 `clusterIP: None` 的 Service 没有 ClusterIP。DNS 直接返回所有后端 Pod 的 IP。
适用于：数据库集群（需要知道每个实例的 IP）、自定义负载均衡。

In [ ]:
%%writefile /tmp/k8s-demo/headless-svc.yaml
apiVersion: v1
kind: Service
metadata:
  name: redis-headless
spec:
  clusterIP: None             # Headless！
  selector:
    app: redis
  ports:
  - port: 6379

print("Headless Service 的 DNS 会返回所有 Pod IP 的 A 记录")
print("适合 StatefulSet 场景：每个 Pod 有独立 DNS")

---

## Ingress：HTTP/HTTPS 路由

Service 只解决了 L4（TCP/UDP）的负载均衡。现实中的 HTTP 应用需要：
- 基于域名/路径路由到不同 Service（`api.example.com` → API Service，`www.example.com` → 前端 Service）
- TLS 证书管理
- URL 重写、限流

**Ingress** 就是解决这些问题的 L7（HTTP）路由。

```
                     ┌──────────────┐
                     │  Ingress     │
                     │  Controller  │
                     │  (nginx)     │
                     └──┬───┬───┬──┘
                        │   │   │
          ┌─────────────┘   │   └─────────────┐
          ▼                 ▼                  ▼
    api.example.com   www.example.com   admin.example.com
    → api-svc:5000    → web-svc:3000    → admin-svc:8000
```

**注意**：Ingress 只是规则定义，需要一个 **Ingress Controller**（如 nginx-ingress）来实现。

In [ ]:
# 在 kind 中安装 nginx-ingress
# ! kubectl apply -f https://raw.githubusercontent.com/kubernetes/ingress-nginx/main/deploy/static/provider/kind/deploy.yaml
# ! sleep 10

print("对于 kind 集群：")
print("  kubectl apply -f https://raw.githubusercontent.com/kubernetes/ingress-nginx/main/deploy/static/provider/kind/deploy.yaml")
print("对于 minikube：")
print("  minikube addons enable ingress")

In [ ]:
# Ingress 配置示例
%%writefile /tmp/k8s-demo/ingress.yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: my-ingress
  annotations:
    nginx.ingress.kubernetes.io/rewrite-target: /
spec:
  rules:
  - host: api.myapp.local         # 这个域名 →
    http:
      paths:
      - path: /
        pathType: Prefix
        backend:
          service:
            name: api-service      # → 转发到这个 Service
            port:
              number: 5000
  - host: admin.myapp.local
    http:
      paths:
      - path: /
        pathType: Prefix
        backend:
          service:
            name: admin-service
            port:
              number: 8000

print("此 Ingress 定义了两个域名路由：")
print("  api.myapp.local → api-service:5000")
print("  admin.myapp.local → admin-service:8000")

---

## 存储：让数据"活过"Pod 的生命周期

Pod 是临时的，但数据必须持久。K8s 存储体系有三层抽象：

```
Pod ──PVC──▶ PV ──▶ 实际存储（本地磁盘 / NFS / 云盘）
     (申请)   (资源)      (后端)
```

| 概念 | 说明 | 类比 |
|------|------|------|
| **Volume** | Pod 级别的临时存储 | 便当盒（Pod 销毁就没了） |
| **PersistentVolume (PV)** | 集群管理员创建的存储资源 | 仓库里的储物柜 |
| **PersistentVolumeClaim (PVC)** | 用户申请存储 | 填申请单要一个柜子 |
| **StorageClass** | 动态创建 PV 的模板 | 自动分配柜子的系统 |

In [ ]:
%%writefile /tmp/k8s-demo/pvc.yaml
# 先定义 PVC（申请存储）
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: my-pvc
spec:
  accessModes:
    - ReadWriteOnce       # 单节点读写
  resources:
    requests:
      storage: 1Gi        # 申请 1GB

! kubectl apply -f /tmp/k8s-demo/pvc.yaml 2>/dev/null
! kubectl get pvc 2>/dev/null

In [ ]:
# 在 Pod 中使用 PVC
%%writefile /tmp/k8s-demo/pod-with-pvc.yaml
apiVersion: v1
kind: Pod
metadata:
  name: storage-demo
spec:
  containers:
  - name: writer
    image: python:3.12-slim
    command: ["python", "-c"]
    args:
    - |
      import os, time
      data_file = "/data/counter.txt"
      os.makedirs("/data", exist_ok=True)
      count = 0
      if os.path.exists(data_file):
          with open(data_file) as f:
              count = int(f.read())
      count += 1
      with open(data_file, "w") as f:
          f.write(str(count))
      print(f"写入计数: {count}")
      time.sleep(3600)
    volumeMounts:
    - name: storage
      mountPath: /data
  volumes:
  - name: storage
    persistentVolumeClaim:
      claimName: my-pvc          # 引用 PVC
  restartPolicy: Never

! kubectl apply -f /tmp/k8s-demo/pod-with-pvc.yaml 2>/dev/null
! sleep 3
! kubectl logs storage-demo 2>/dev/null

In [ ]:
# 验证持久化：删除 Pod 再创建，数据还在
! kubectl delete pod storage-demo --wait=true 2>/dev/null
! kubectl apply -f /tmp/k8s-demo/pod-with-pvc.yaml 2>/dev/null
! sleep 3
! kubectl logs storage-demo 2>/dev/null
print("如果计数器从 2 开始，说明数据持久化了！")

In [ ]:
# 清理
! kubectl delete pod storage-demo --wait=false 2>/dev/null
! kubectl delete pvc my-pvc --wait=false 2>/dev/null

---

## StatefulSet：有状态应用

Deployment 适合无状态应用（Web、API）。对于数据库、消息队列等有状态应用，需要：
- 稳定的网络标识（Pod 名可预测：`redis-0`, `redis-1`, `redis-2`）
- 稳定的持久化存储（每个 Pod 有自己的 PVC，不共享）
- 有序的启动和停止（0 → 1 → 2，停止时 2 → 1 → 0）

| 对比 | Deployment | StatefulSet |
|------|------------|-------------|
| Pod 名称 | 随机后缀（py-api-5d8f-abc123） | 有序索引（redis-0） |
| 网络标识 | 不稳定 | 稳定 DNS（redis-0.redis-svc.default.svc） |
| 存储 | 共享或共享 PVC 模板 | 每个 Pod 独立 PVC |
| 启停顺序 | 并行 | 有序（0,1,2...） |
| 适用 | 无状态 Web/API | 数据库、消息队列、ZK |

In [ ]:
%%writefile /tmp/k8s-demo/statefulset.yaml
apiVersion: v1
kind: Service
metadata:
  name: redis-svc
spec:
  clusterIP: None            # Headless，配合 StatefulSet
  selector:
    app: redis-sts
  ports:
  - port: 6379
---
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: redis-sts
spec:
  serviceName: redis-svc     # 必须指定 Headless Service
  replicas: 3
  selector:
    matchLabels:
      app: redis-sts
  template:
    metadata:
      labels:
        app: redis-sts
    spec:
      containers:
      - name: redis
        image: redis:7-alpine
        ports:
        - containerPort: 6379
  # volumeClaimTemplates: 每个 Pod 自动创建独立 PVC
  # 省略以简化演示，实际使用时应该加上

! kubectl apply -f /tmp/k8s-demo/statefulset.yaml 2>/dev/null
! sleep 5
! kubectl get pods -l app=redis-sts 2>/dev/null
print("\n注意 Pod 名称：redis-sts-0, redis-sts-1, redis-sts-2（有序！）")

In [ ]:
# 验证稳定网络标识：每个 Pod 有独立 DNS
! kubectl run dns-test --image=busybox --rm -it --restart=Never -- \
  nslookup redis-sts-0.redis-svc 2>/dev/null || echo "DNS 测试完成"

# 清理
! kubectl delete statefulset redis-sts --wait=false 2>/dev/null
! kubectl delete service redis-svc --wait=false 2>/dev/null

---

## NetworkPolicy：防火墙规则

默认情况下，K8s 中所有 Pod 可以互相通信。这在安全上非常危险——攻击者拿下一个 Web Pod，就能直接访问数据库。

**NetworkPolicy** 定义 Pod 间的流量规则："谁可以访问谁"。

**注意**：NetworkPolicy 需要 CNI 插件支持（Calico、Cilium 支持；Flannel 默认不支持）。

In [ ]:
%%writefile /tmp/k8s-demo/network-policy.yaml
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: db-policy
spec:
  podSelector:
    matchLabels:
      app: database         # 规则应用于数据库 Pod
  policyTypes:
  - Ingress                 # 入站规则
  ingress:
  - from:
    - podSelector:
        matchLabels:
          role: backend      # 只允许后端 Pod 访问
    ports:
    - protocol: TCP
      port: 5432

print("此策略表示：")
print("  只有带 label 'role: backend' 的 Pod 可以访问数据库的 5432 端口")
print("  其他 Pod（包括外部）都无法访问数据库")

---

## 🎯 第2周总结

| 概念 | 一句话 |
|------|--------|
| **K8s 网络模型** | 所有 Pod 可互通，CNI 插件实现 |
| **CoreDNS** | Service 名自动解析为 ClusterIP |
| **Ingress** | L7 HTTP 路由：域名/路径 → Service |
| **PV/PVC** | 存储抽象：申请(PVC) → 绑定(PV) → 使用 |
| **StatefulSet** | 有状态应用的 Deployment：有序、稳定标识、独立存储 |
| **NetworkPolicy** | Pod 间防火墙，最小权限原则 |

---

## 🧪 综合练习

为你的应用做好网络和存储规划：

1. **Ingress** 配置两个域名路由
2. **Web 服务**使用 PVC 存储用户上传的文件
3. **数据库**使用 StatefulSet + PVC（数据不丢）
4. **NetworkPolicy**：DB 只允许 Web 和 Worker 访问
5. **Secret** 管理所有密码

In [ ]:
# 你的练习
pass